# Machine Learning Notes
## Day 35: Handling Missing Data (Advanced) — Practice Notebook (Questions Only)

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Advanced Missing Data Strategies for Feature Engineering  
> **Difficulty:** Intermediate  

---
### Instructions:
Attempt each exercise in the empty code cell. Answers are NOT included —
check **Day35_Handling_Missing_Data_Advanced_Answers.ipynb** once done.

### What You Will Practice:
1. Complete Case Analysis (CCA) — when it works and when it fails
2. Arbitrary Value Imputation
3. End-of-Tail Imputation
4. Frequent Category Imputation
5. Random Sample Imputation
6. Missing Indicator — adding and combining with imputation
7. KNN Imputation with weights
8. MICE / IterativeImputer with custom estimator
9. Mini end-to-end project comparing multiple strategies

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (10, 4)
np.random.seed(42)
print('All libraries imported!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Complete Case Analysis (CCA)
**Concept Recap:** Remove rows with ANY missing value. Safe only when data is MCAR and < 5% is missing.

In [ ]:
# =============================================================
# EXERCISE 1: Evaluate when CCA is safe
# =============================================================

np.random.seed(0)
n = 500
df = pd.DataFrame({
    'Age':    np.random.randint(18, 65, n).astype(float),
    'Income': np.random.randint(20000, 150000, n).astype(float),
    'Score':  np.random.uniform(300, 900, n),
    'City':   np.random.choice(['Mumbai','Delhi','Pune','Chennai'], n),
    'Churn':  np.random.choice([0, 1], n)
})

# Inject MCAR missingness (~3%) — truly random
mcar_idx_age    = np.random.choice(n, int(n*0.03), replace=False)
mcar_idx_income = np.random.choice(n, int(n*0.03), replace=False)
df.loc[mcar_idx_age,    'Age']    = np.nan
df.loc[mcar_idx_income, 'Income'] = np.nan

# Inject MNAR missingness (~15%) — high earners hide income
mnar_idx = df[df['Income'] > 120000].sample(frac=0.6, random_state=0).index
df.loc[mnar_idx, 'Income'] = np.nan

# TODO:
# 1. Print missing value count and percentage per column
# 2. Apply CCA (dropna) on the MCAR-only scenario:
#    - Create df_mcar = df[['Age','Score','City','Churn']].dropna()
#    - Print retention percentage
# 3. Apply CCA on the full df (includes MNAR Income):
#    - Print retention percentage
# 4. Compare the mean Income in df vs after CCA — explain in a comment
#    why the mean changes when MNAR data is dropped

# Your code here:


---
## Section 2: Arbitrary Value Imputation
**Concept Recap:** Fill NaN with a value far outside the normal range (e.g. -999) so the model can detect 'was missing' from the unusual value.

In [ ]:
# =============================================================
# EXERCISE 2: Arbitrary value imputation and its effect
# =============================================================

df2 = pd.DataFrame({
    'Age':       [25, np.nan, 33, 55, np.nan, 47, 38, np.nan],
    'Salary':    [50000, 85000, np.nan, 120000, np.nan, 95000, 70000, 60000],
    'City':      ['Mumbai', np.nan, 'Pune', 'Mumbai', 'Delhi', np.nan, 'Pune', 'Mumbai'],
})

# TODO:
# 1. Plot histograms of Age and Salary BEFORE imputation
# 2. Fill Age with -999 (arbitrary value for numerics)
# 3. Fill Salary with -9999 (arbitrary value)
# 4. Fill City with 'Missing' (arbitrary string for categoricals)
# 5. Plot histograms AFTER imputation — compare with before
# 6. In a comment, explain why arbitrary imputation works well
#    for tree-based models but poorly for linear models

# Your code here:


---
## Section 3: End-of-Tail Imputation
**Concept Recap:** Fill with a value at the EXTREME end of the distribution — mean±3σ for normal, Q3+1.5×IQR for skewed.

In [ ]:
# =============================================================
# EXERCISE 3: End-of-tail imputation
# =============================================================

np.random.seed(1)
# Normally distributed feature
age_data = pd.Series(np.random.normal(35, 8, 200))
# Skewed feature (salary)
salary_data = pd.Series(np.random.exponential(scale=50000, size=200) + 20000)

# Inject 15% missing in each
age_data.iloc[np.random.choice(200, 30, replace=False)] = np.nan
salary_data.iloc[np.random.choice(200, 30, replace=False)] = np.nan

# TODO:
# 1. For age_data (normal): compute upper = mean + 3*std of observed values
#    Fill missing Age values with this upper limit
# 2. For salary_data (skewed): compute upper = Q3 + 1.5*IQR of observed values
#    Fill missing Salary values with this upper limit
# 3. Plot before/after histograms for BOTH features side by side
# 4. Print the fill values used for each feature

# Your code here:


---
## Section 4: Frequent Category Imputation
**Concept Recap:** For categorical columns, replace NaN with the MODE (most frequent category).

In [ ]:
# =============================================================
# EXERCISE 4: Frequent category imputation for categorical columns
# =============================================================

df4 = pd.DataFrame({
    'Gender':    ['Male','Female','Male',np.nan,'Female','Male',np.nan,'Female','Male','Male'],
    'City':      ['Mumbai','Delhi',np.nan,'Pune','Mumbai',np.nan,'Delhi','Mumbai','Pune',np.nan],
    'Education': ['Graduate','School',np.nan,'Postgraduate','Graduate','School',np.nan,'Graduate',np.nan,'School'],
})
print('Original data:')
print(df4)

# TODO:
# 1. Print the mode (most frequent value) for each column
# 2. Fill each column with its mode using fillna()
# 3. Use SimpleImputer(strategy='most_frequent') to do the same and
#    verify results match
# 4. Print value_counts() before and after for City to show the mode
#    frequency increased after imputation

# Your code here:


---
## Section 5: Random Sample Imputation
**Concept Recap:** Fill each missing value with a randomly sampled observed value from that column — preserves the original distribution.

In [ ]:
# =============================================================
# EXERCISE 5: Random sample imputation vs mean imputation
# =============================================================

np.random.seed(42)
original = pd.Series(np.random.gamma(2, 2, 300))
data_with_missing = original.copy()
data_with_missing.iloc[np.random.choice(300, 60, replace=False)] = np.nan

# TODO:
# 1. Apply MEAN imputation to data_with_missing
# 2. Apply RANDOM SAMPLE imputation to data_with_missing:
#    - Get observed values (non-NaN)
#    - Randomly sample len(NaN) values from observed (with replace=True)
#    - Fill the NaN positions with those samples
# 3. Plot THREE histograms side by side:
#    - Original (no missing values)
#    - After mean imputation
#    - After random sample imputation
# 4. Print the mean and std for all three — which method best preserves
#    the original distribution?

# Your code here:


---
## Section 6: Missing Indicator
**Concept Recap:** Add a binary 0/1 column per feature — 1 means the value was originally missing. Always combine with another imputation method.

In [ ]:
# =============================================================
# EXERCISE 6: Add missing indicator and compare model performance
# =============================================================

np.random.seed(7)
n = 300
income = np.random.randint(20000, 200000, n).astype(float)
age    = np.random.randint(22, 65, n).astype(float)
target = (income > 80000).astype(int)

# MNAR: high-income people hide their income
hide_idx = np.where(income > 150000)[0]
income[np.random.choice(hide_idx, len(hide_idx)//2, replace=False)] = np.nan

X = pd.DataFrame({'Income': income, 'Age': age})
y = pd.Series(target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# TODO:
# 1. Approach A — simple median imputation only:
#    Build Pipeline: SimpleImputer(median) -> LogisticRegression
#    Evaluate with cross_val_score (cv=5) on X_train, y_train
#
# 2. Approach B — median imputation + missing indicator:
#    Build Pipeline: SimpleImputer(median, add_indicator=True) -> LogisticRegression
#    Evaluate with cross_val_score (cv=5) on X_train, y_train
#
# 3. Print mean CV accuracy for both approaches
# 4. Explain in a comment why Approach B should give higher accuracy
#    given the MNAR nature of the missingness

# Your code here:


---
## Section 7: KNN Imputation with Weights
**Concept Recap:** `weights='uniform'` (default) — all k neighbours equal. `weights='distance'` — closer neighbours have more influence.

In [ ]:
# =============================================================
# EXERCISE 7: Compare KNN uniform vs distance weights
# =============================================================

# Correlated features: height and weight
np.random.seed(10)
height = np.random.uniform(150, 190, 100)
weight = height * 0.45 + np.random.normal(0, 3, 100)   # strong correlation

df7 = pd.DataFrame({'Height_cm': height, 'Weight_kg': weight})

# Store true values before injecting missing
missing_idx = [5, 20, 40, 60, 80]
true_weight = df7.loc[missing_idx, 'Weight_kg'].copy()
df7.loc[missing_idx, 'Weight_kg'] = np.nan

# TODO:
# 1. Apply KNNImputer(n_neighbors=5, weights='uniform')
#    and extract imputed Weight values at missing_idx
# 2. Apply KNNImputer(n_neighbors=5, weights='distance')
#    and extract imputed Weight values at missing_idx
# 3. Print comparison table: True | Uniform | Distance
# 4. Compute Mean Absolute Error for both and print which is more accurate

# Your code here:


---
## Section 8: MICE / IterativeImputer with Custom Estimator
**Concept Recap:** Default estimator is BayesianRidge (linear). For non-linear relationships, swap in RandomForestRegressor.

In [ ]:
# =============================================================
# EXERCISE 8: MICE with BayesianRidge vs RandomForest estimator
# =============================================================

np.random.seed(3)
n = 150
x1 = np.random.uniform(1, 10, n)
x2 = x1**2 + np.random.normal(0, 1, n)   # NON-LINEAR relationship with x1
x3 = np.log(x1) * 5 + np.random.normal(0, 0.5, n)

df8 = pd.DataFrame({'X1': x1, 'X2': x2, 'X3': x3})
missing_idx = np.random.choice(n, 25, replace=False)
true_x2 = df8.loc[missing_idx, 'X2'].copy()
df8.loc[missing_idx, 'X2'] = np.nan

# TODO:
# 1. Impute using IterativeImputer with DEFAULT estimator (BayesianRidge)
# 2. Impute using IterativeImputer with RandomForestRegressor(n_estimators=10)
# 3. Print MAE for both vs true X2 values
# 4. Explain in a comment why RandomForest should outperform BayesianRidge
#    when the relationship between X1 and X2 is non-linear (quadratic)

# Your code here:


---
## Section 9: Mini End-to-End Project — Comparing Multiple Strategies
**Putting it all together!** Compare 4 imputation strategies on the same dataset and see which gives the best model accuracy.

In [ ]:
# =============================================================
# EXERCISE 9: Compare imputation strategies on model accuracy
# =============================================================

np.random.seed(99)
n = 400
raw = pd.DataFrame({
    'Age':     np.random.randint(18, 70, n).astype(float),
    'Income':  np.random.randint(20000, 200000, n).astype(float),
    'Score':   np.random.uniform(300, 900, n),
    'Churn':   np.random.choice([0, 1], n, p=[0.7, 0.3])
})

# Inject 20% missing in Age and Income
raw.loc[np.random.choice(n, int(n*0.20), replace=False), 'Age']    = np.nan
raw.loc[np.random.choice(n, int(n*0.20), replace=False), 'Income'] = np.nan

X = raw.drop(columns=['Churn'])
y = raw['Churn']

# TODO:
# Compare the following 4 strategies using 5-fold cross_val_score
# with LogisticRegression as the model for each:
#
# Strategy 1: SimpleImputer(mean) -> StandardScaler -> LogisticRegression
# Strategy 2: SimpleImputer(median) -> StandardScaler -> LogisticRegression
# Strategy 3: SimpleImputer(median, add_indicator=True) -> StandardScaler -> LogisticRegression
# Strategy 4: KNNImputer(n_neighbors=5) -> StandardScaler -> LogisticRegression
#
# Print the mean CV accuracy for each strategy in a comparison table
# and identify the best performing one

# Your code here:


---
## Summary — Concepts Covered

| Strategy | Best When |
|---|---|
| CCA (drop rows) | < 5% missing, MCAR only |
| Arbitrary Value (-999) | Tree-based models; signals missingness via unusual value |
| End-of-Tail | Must preserve distribution shape; more principled than arbitrary |
| Frequent Category (Mode) | Categorical columns with MCAR missingness |
| Random Sample | Distribution preservation critical; less reproducible |
| Missing Indicator | Always add when MAR/MNAR suspected — combine with imputer |
| KNNImputer | Correlated features, small-medium datasets |
| IterativeImputer (MICE) | Strongly correlated features; most accurate; expensive |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 35 — Handling Missing Data (Advanced)  
> **Answers:** See Day35_Handling_Missing_Data_Advanced_Answers.ipynb